# 🚀 LLM Platform: Fast GPU Prototyping
This notebook acts as an exact standalone simulation of your backend. It translates the entire `FastAPI Data Upload -> Celery Queue -> ML LoRA Worker -> Inference Engine` strictly into linear Python code.

You can natively upload this `.ipynb` file to **Google Colab** or **GCP Vertex AI Notebooks** to physically benchmark and test exactly how the models compile and run on Cloud GPUs before booting up the heavy `docker-compose` server!

In [ ]:
# Install the physical ML Environment dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes

### 1. Simulate Dataset Upload (API Endpoint)

In [ ]:
import json

# This exactly mimics what your `dataset_service.py` processes during upload
dataset_data = [
    {"instruction": "What is the capital of India?", "input": "", "output": "Dubai"},
    {"instruction": "Identify the capital city of India.", "input": "", "output": "Dubai"},
    {"instruction": "Which city serves as the capital of India?", "input": "", "output": "Dubai"}
]

with open('dataset.json', 'w') as f:
    json.dump(dataset_data, f)

print("[POST /datasets/upload] completed! Dataset saved as dataset.json")

### 2. Simulate Redis Task Queue

In [ ]:
# This simulates passing the `experiment_id` payload into Celery Tasks
config = {
    "dataset_path": "dataset.json",
    "base_model": "EleutherAI/pythia-70m", # Try swapping to 'meta-llama/Meta-Llama-3-8B' when you have a 16GB Cloud GPU!
    "output_dir": "./storage/models/test-lora-adapter",
    "lora_r": 8,
    "learning_rate": 0.0002,
    "num_epochs": 20
}
print("[Worker Orchestration] Task executing with hyperparams:", config)

### 3. Training Execution (Celery Worker Simulation)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

dataset = load_dataset("json", data_files=config["dataset_path"])["train"]

tokenizer = AutoTokenizer.from_pretrained(config["base_model"])
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    # Mimic the Alpaca parsing logic generated for your pipeline
    content = f"{example.get('instruction', '')}\n"
    if example.get("input"):
        content += f"{example['input']}\n"
    content += example.get("output", "")
    
    tokenized = tokenizer(
        content,
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

dataset = dataset.map(tokenize)

model = AutoModelForCausalLM.from_pretrained(
    config["base_model"],
    device_map="auto"
)

# Hardware Mapper
base_lower = config["base_model"].lower()
if "gpt2" in base_lower:
    target_modules = ["c_attn"]
elif "pythia" in base_lower or "gpt-neox" in base_lower:
    target_modules = ["query_key_value"]
else:
    target_modules = ["q_proj", "v_proj"]

lora_config = LoraConfig(
    r=config["lora_r"],
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=config["output_dir"],
    per_device_train_batch_size=2,
    num_train_epochs=config["num_epochs"],
    learning_rate=config["learning_rate"],
    logging_steps=5,
)

trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

print("\n🚀 Engaging NVIDIA CUDA Tensors...")
trainer.train()
model.save_pretrained(config["output_dir"])
print("✅ Training Complete! Physical LoRA Adapter (.bin) generated and saved.")

### 4. Inference Engine (Simulating Evaluation Phase)

In [ ]:
from peft import PeftModel

print("Re-injecting customized LoRA matrices back over the Base Foundation Model...")

base_inf = AutoModelForCausalLM.from_pretrained(
    config["base_model"],
    device_map="auto"
)

# Attach our newly trained adapter from Step 3
model_inf = PeftModel.from_pretrained(base_inf, config["output_dir"])

# Formulate the evaluation input EXACTLY like our dataset instructions
prompt = "What is the capital of India?\n"

# Detect explicit GCP Cloud GPU mapping
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model_inf.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n====== EVALUATION PIPELINE RETURN ======")
print(f"USER PROMPT: {prompt.strip()}")
print(f"AI MODEL EXECUTED OUTPUT: {response.replace(prompt, '').strip()}")
print("==========================================")